# Exam Practice - AlpesHearth

## Exploración de datos 

Verifico que se cargue correctamente el DataFrame y reviso sus dimensiones

In [ ]:
import pandas as pd

df = pd.read_csv("../data/Datos Lab 1.csv")
pd.set_option('display.max_columns', None)
print(df.head()) # Verificar que el archivo se cargo correctamente
print(df.tail())
print(df.shape[0]) #Número de filas del df
print(df.shape[1]) #Número de columnas de 
print(df.shape) #Dimensiones del df

  Patient ID    Date of Service Sex   Age  Weight (kg)  Height (m)     BMI  \
0   isDx5313  November 08, 2023   M  44.0      114.300       1.720  38.600   
1   LHCK2961         20/03/2024   F  57.0       92.923       1.842  33.116   
2   WjVn1699         2021-05-27   F   NaN       73.400       1.650  27.000   
3   dCDO1109     April 18, 2022   F  35.0      113.300       1.780  35.800   
4   pnpE1080         01/11/2024   F  48.0      102.200       1.750  33.400   

   Abdominal Circumference (cm) Blood Pressure (mmHg)  \
0                       100.000                112/83   
1                       106.315                101/91   
2                        78.100                 90/74   
3                        79.600                 92/89   
4                       106.700                121/68   

   Total Cholesterol (mg/dL)  HDL (mg/dL)  Fasting Blood Sugar (mg/dL)  \
0                      228.0         77.0                         91.0   
1                      158.0         71.

Exploro variables númericas y categoricas para ver si coinciden con el diccionario, y hago un primer acercamiento sobre los valores nulos 

In [19]:
df.info()
variables_numericas = df.select_dtypes(include=['int64', 'float64']).columns.to_list()
variables_categoricas = df.select_dtypes(include=['object']).columns.to_list()
print(f"Variables númericas: {len(variables_numericas)}")
print(variables_numericas)
print(f"Variables categoricas: {len(variables_categoricas)}")
print(variables_categoricas)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1639 entries, 0 to 1638
Data columns (total 24 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Patient ID                    1639 non-null   object 
 1   Date of Service               1639 non-null   object 
 2   Sex                           1639 non-null   object 
 3   Age                           1571 non-null   float64
 4   Weight (kg)                   1566 non-null   float64
 5   Height (m)                    1578 non-null   float64
 6   BMI                           1586 non-null   float64
 7   Abdominal Circumference (cm)  1578 non-null   float64
 8   Blood Pressure (mmHg)         1639 non-null   object 
 9   Total Cholesterol (mg/dL)     1571 non-null   float64
 10  HDL (mg/dL)                   1557 non-null   float64
 11  Fasting Blood Sugar (mg/dL)   1585 non-null   float64
 12  Smoking Status                1639 non-null   object 
 13  Dia

Exploro nulos, duplicados (completos, lógicos e inconsistencias) y outliers

In [ ]:
# Exploración de nulos

valores_nulos_por_categoria = df.isnull().sum()
porcentaje_nulos_por_categoria = ((valores_nulos_por_categoria / len(df)) * 100).round(2)

nulos = pd.DataFrame({
    "Nulos": valores_nulos_por_categoria,
    "Porcentaje (%)": porcentaje_nulos_por_categoria
})

print(nulos[nulos['Nulos'] > 0])

# Exploración de duplicados identicos 

duplicados_identicos = df.duplicated().sum()
print(f"Duplicados identidenticos: {duplicados_identicos}")

# Exploración de duplicados por id 

duplicados_por_id_cantidad = df.duplicated(subset=['Patient ID']).sum()
print(f"Duplicados por ID: {duplicados_por_id_cantidad}")

duplicados_por_id = df[df['Patient ID'].duplicated(keep=False)]
print(duplicados_por_id.sort_values('Patient ID').head(10))

# En este caso no es recomendable eliminar duplicados lógicos, pues pueden haber 2 pacientes con las mismas estadisticas

# Outliers 

for col in variables_numericas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
    print(f"{col}: {len(outliers)} outliers")

                              Nulos  Porcentaje (%)
Age                              68            4.15
Weight (kg)                      73            4.45
Height (m)                       61            3.72
BMI                              53            3.23
Abdominal Circumference (cm)     61            3.72
Total Cholesterol (mg/dL)        68            4.15
HDL (mg/dL)                      82            5.00
Fasting Blood Sugar (mg/dL)      54            3.29
Height (cm)                      68            4.15
Waist-to-Height Ratio            76            4.64
Systolic BP                      61            3.72
Diastolic BP                     85            5.19
Estimated LDL (mg/dL)            57            3.48
CVD Risk Score                   29            1.77
Duplicados identidenticos: 151
Duplicados por ID: 263
     Patient ID Date of Service Sex   Age  Weight (kg)  Height (m)     BMI  \
17     AhYt1346      09-28-2020   M  41.0       71.300       1.730  23.800   
1584   AhY

Exploro valores imposibles, dado que es un caso clinico, además de los outliers, es importante revisar si existen valores imposibles.

In [40]:
edad = (df['Age'] < 18).sum()
peso = (df['Weight (kg)'] < 30).sum()
bmi = (df['BMI'] < 10).sum()
ldl = (df['Estimated LDL (mg/dL)'] < 0).sum()
print(f"Cantidad de valores imposibles de edad: {edad}")
print(f"Cantidad de valores imposibles de peso: {peso}")
print(f"Cantidad de valores imposibles de bmi: {bmi}")
print(f"Cantidad de valores imposibles de ldl: {ldl}")

Cantidad de valores imposibles de edad: 10
Cantidad de valores imposibles de peso: 8
Cantidad de valores imposibles de bmi: 7
Cantidad de valores imposibles de ldl: 16


## Preparación de datos 

En primera instancia, elimino duplicados identicos, e inconsistencias relacionadas con los IDs duplicados 

In [43]:
df_clean = df.copy()
before = len(df_clean)
df_clean = df_clean.drop_duplicates(keep='first')
after = len(df_clean)

print(f"Cantidad de registros eliminados: {before - after}")

df_clean = df_clean.drop_duplicates(subset=['Patient ID'], keep=False)

duplicados = df_clean.duplicated().sum()
IDs_duplicados = df_clean.duplicated(subset=['Patient ID']).sum()

print(f"Duplicados Identicos: {duplicados}")
print(f"Pacientes con IDs duplicados: {IDs_duplicados}")
print(f"\nDimensiones despues de eliminar duplicados: {df_clean.shape}")


Cantidad de registros eliminados: 151
Duplicados Identicos: 0
Pacientes con IDs duplicados: 0

Dimensiones despues de eliminar duplicados: (1264, 24)


Posteriormente, transformo la columna 'Age' de float a Integer, como indica el diccionario, convierto outliers a nulos para que sean imputados por la mediana en el pipeline y elimino valores imposibles 

In [51]:
import numpy as np

df_clean['Age'] = df_clean['Age'].round().astype('Int64')
print(df_clean['Age'].head(5))

for col in variables_numericas:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    df_clean[col] = df_clean[col].where(
        (df_clean[col] >= limite_inf) & (df_clean[col] <= limite_sup), other=np.nan
    )
    outliers = df_clean[(df_clean[col] < limite_inf) | (df_clean[col] > limite_sup)]
    print(f"{col}: {len(outliers)} outliers")
    
df_clean = df_clean[df_clean['Age'] >= 18]
df_clean = df_clean[df_clean['Weight (kg)'] >= 30]
df_clean = df_clean[df_clean['BMI'] >= 10]
df_clean = df_clean[df_clean['Estimated LDL (mg/dL)'] >= 0]
df_clean = df_clean[df_clean['CVD Risk Score'] >= 0]

valores_imposibles_edad = (df_clean['Age'] < 18).sum()
valores_imposibles_peso = (df_clean['Weight (kg)'] < 30).sum()
valores_imposibles_bmi = (df_clean['BMI'] < 10).sum()
valores_imposibles_ldl = (df_clean['Estimated LDL (mg/dL)'] < 0).sum()
valores_imposibles_cvdrisk = (df_clean['CVD Risk Score'] < 0).sum()

print(f"Valores imposibles edad: {valores_imposibles_edad}")
print(f"Valores imposibles peso: {valores_imposibles_peso}")
print(f"Valores imposibles bmi: {valores_imposibles_bmi}")
print(f"Valores imposibles ldl: {valores_imposibles_ldl}")
print(f"Valores imposibles cvd risk score: {valores_imposibles_cvdrisk}")


0    44
1    57
3    35
4    48
5    43
Name: Age, dtype: Int64
Age: 0 outliers
Weight (kg): 0 outliers
Height (m): 0 outliers
BMI: 0 outliers
Abdominal Circumference (cm): 0 outliers
Total Cholesterol (mg/dL): 0 outliers
HDL (mg/dL): 0 outliers
Fasting Blood Sugar (mg/dL): 0 outliers
Height (cm): 0 outliers
Waist-to-Height Ratio: 0 outliers
Systolic BP: 0 outliers
Diastolic BP: 0 outliers
Estimated LDL (mg/dL): 0 outliers
CVD Risk Score: 0 outliers
Valores imposibles edad: 0
Valores imposibles peso: 0
Valores imposibles bmi: 0
Valores imposibles ldl: 0
Valores imposibles cvd risk score: 0


## Construcción del pipeline 

Hago drop de las columnas redundantes, imputo nulos númericos con la mediana y nulos categoricos con la moda, y transformo las variables categoricas que deberian ser boolean con el OneHotEncoder 

In [54]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer

target = 'CVD Risk Score'
X = df_clean.drop(columns=[target])
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

numeric_features = ["Age",
                    "Weight (kg)","Height (m)",
                    "BMI",
                    "Abdominal Circumference (cm)",
                    "Total Cholesterol (mg/dL)",
                    "HDL (mg/dL)",
                    "Fasting Blood Sugar (mg/dL)",
                    "Waist-to-Height Ratio",
                    "Systolic BP",
                    "Diastolic BP",
                    "Estimated LDL (mg/dL)"
]

categorical_features = ["Sex","Smoking Status","Diabetes Status","Physical Activity Level",
                        "Family History of CVD","Blood Pressure Category"]

print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)

columns_to_drop = [
    'Patient ID',
    'Blood Pressure (mmHg)',
    'Height (cm)',
    'CVD Risk Level',
    'Date of Service'
]

def drop_columns(df):
    return df.drop(columns=columns_to_drop, errors='ignore')

dropper = FunctionTransformer(drop_columns)

numerical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy='median')),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("onehot", OneHotEncoder(handle_unknown='ignore', drop='if_binary'))
])

preprocessor = ColumnTransformer([
    ("num", numerical_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

pipeline = Pipeline([
    ("dropper", dropper),
    ("preprocesamiento", preprocessor),
    ("modelo", LinearRegression())
])

pipeline.fit(X_train, y_train) #Entrenamiento

#Predicción 
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

print("Métricas sobre TRAIN:")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_train, y_train_pred):.4f}")
print(f"R²: {r2_score(y_train, y_train_pred):.4f}")

print("\nMétricas sobre TEST:")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_test_pred):.4f}")
print(f"R²: {r2_score(y_test, y_test_pred):.4f}")

Numéricas: ['Age', 'Weight (kg)', 'Height (m)', 'BMI', 'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP', 'Estimated LDL (mg/dL)']
Categóricas: ['Sex', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD', 'Blood Pressure Category']
Métricas sobre TRAIN:
RMSE: 0.3831
MAE: 0.1376
R²: 0.9766

Métricas sobre TEST:
RMSE: 0.2170
MAE: 0.1102
R²: 0.9922
